In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/paimana_train_v1.parquet")

print("Shape:", df.shape)
print(df.head())

Shape: (109787, 52)
  project_id report_month  month  year sector           state  approval_year  \
0  120100067   2015-04-01      4  2015  STEEL  ANDHRA PRADESH           2005   
1  120100067   2015-05-01      5  2015  STEEL  ANDHRA PRADESH           2005   
2  120100067   2015-06-01      6  2015  STEEL  ANDHRA PRADESH           2005   
3  120100067   2015-07-01      7  2015  STEEL  ANDHRA PRADESH           2005   
4  120100067   2015-08-01      8  2015  STEEL  ANDHRA PRADESH           2005   

   project_age  original_duration_months  duration_overrun_months  ...  \
0        114.0                      48.0                     66.0  ...   
1        115.0                      48.0                     67.0  ...   
2        116.0                      48.0                     68.0  ...   
3        117.0                      48.0                     69.0  ...   
4        118.0                      48.0                     70.0  ...   

   cost_rebaseline_signal  schedule_progress_mismatch 

In [2]:
print(df.dtypes)

project_id                                        str
report_month                           datetime64[us]
month                                           int64
year                                            int64
sector                                            str
state                                             str
approval_year                                   int64
project_age                                   float64
original_duration_months                      float64
duration_overrun_months                       float64
original_cost_crore                           float64
revised_cost_crore                            float64
anticipated_cost_crore                        float64
current_cost                                  float64
cumulative_expenditure_crore                  float64
anticipated_delay_from_original               float64
remaining_org_months                          float64
delay_revised_months                          float64
milestones_achieved         

In [3]:
print("Cost target:")
print(df["target_cost_overrun_12m"].value_counts(dropna=False))

print("\nSchedule target:")
print(df["target_schedule_risk_12m"].value_counts(dropna=False))

print("\nCost target valid:")
print(df["cost_target_valid"].value_counts(dropna=False))

Cost target:
target_cost_overrun_12m
0.0    77759
NaN    25317
1.0     6711
Name: count, dtype: int64

Schedule target:
target_schedule_risk_12m
NaN    52921
0.0    41249
1.0    15617
Name: count, dtype: int64

Cost target valid:
cost_target_valid
1    84470
0    25317
Name: count, dtype: int64


In [4]:
print(df[
    ["target_cost_overrun_12m",
     "target_schedule_risk_12m",
     "cost_target_valid"]
].isna().sum())

target_cost_overrun_12m     25317
target_schedule_risk_12m    52921
cost_target_valid               0
dtype: int64


In [8]:
def get_walk_forward_group_splits(df, n_folds=5):
    """
    Time-based walk-forward splits.
    Training uses earlier months.
    Validation uses a later month.
    """

    df = df.sort_values("report_month").reset_index(drop=True)

    unique_months = sorted(df["report_month"].dropna().unique())

    # Use the last n_folds months as validation periods
    validation_months = unique_months[-n_folds:]

    splits = []

    for val_month in validation_months:

        train_mask = df["report_month"] < val_month
        val_mask = df["report_month"] == val_month

        train_idx = df.index[train_mask].to_numpy()
        val_idx = df.index[val_mask].to_numpy()

        # Skip empty splits
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        splits.append((train_idx, val_idx))

    return splits

In [6]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score
)

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    """
    Evaluate binary risk predictions.
    """

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    # Remove missing labels
    mask = ~pd.isna(y_true)
    y_true = y_true[mask].astype(int)
    y_prob = y_prob[mask]

    y_pred = (y_prob >= threshold).astype(int)

    # AUC-PR requires both classes
    if len(np.unique(y_true)) == 2:
        auc_pr = average_precision_score(y_true, y_prob)
    else:
        auc_pr = np.nan

    brier = brier_score_loss(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "AUC-PR": auc_pr,
        "Brier Score": brier,
        "F1": f1
    }

In [12]:
cost_results = []

for fold, (train_idx, val_idx) in enumerate(
    get_walk_forward_group_splits(df, n_folds=5),
    start=1
):

    train = df.iloc[train_idx].copy()
    val = df.iloc[val_idx].copy()

    # Remove rows where cost target is unavailable
    train = train[train["target_cost_overrun_12m"].notna()]
    val = val[val["target_cost_overrun_12m"].notna()]

    # Skip fold if there is no validation data
    if len(train) == 0 or len(val) == 0:
        print(f"Skipping Fold {fold}: no valid cost-target data")
        continue

    y_train = train["target_cost_overrun_12m"].astype(int)
    y_val = val["target_cost_overrun_12m"].astype(int)

    # Majority-class prediction
    majority_class = y_train.mode()[0]

    y_pred = np.full(len(y_val), majority_class)

    # Probability of positive class
    positive_probability = (y_train == 1).mean()

    y_prob = np.full(len(y_val), positive_probability)

    print(
        f"Fold {fold}: "
        f"train={len(train)}, "
        f"val={len(val)}, "
        f"majority={majority_class}"
    )

Fold 1: train=84318, val=78, majority=0
Fold 2: train=84396, val=60, majority=0
Fold 3: train=84456, val=11, majority=0
Fold 4: train=84467, val=3, majority=0
Skipping Fold 5: no valid cost-target data


In [15]:
if y_val.nunique() < 2:
    auc_pr = np.nan
else:
    auc_pr = average_precision_score(y_val, y_prob)

In [16]:
fold_result = {
    "fold": fold,
    "accuracy": accuracy_score(y_val, y_pred),
    "f1": f1_score(y_val, y_pred, zero_division=0),
    "auc_pr": auc_pr,
    "brier": brier_score_loss(y_val, y_prob)
}

cost_results.append(fold_result)

In [17]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    average_precision_score,
    brier_score_loss
)

fold_result = {
    "fold": fold,
    "accuracy": accuracy_score(y_val, y_pred),
    "f1": f1_score(y_val, y_pred, zero_division=0),
    "auc_pr": average_precision_score(y_val, y_prob),
    "brier": brier_score_loss(y_val, y_prob)
}

cost_results.append(fold_result)

C:\Users\vatsh\OneDrive\Desktop\ml\meowwww\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [18]:
cost_results_df = pd.DataFrame(cost_results)

print(cost_results_df)
print("\nMean:")
print(cost_results_df.mean(numeric_only=True))

   fold  accuracy   f1  auc_pr     brier
0     5       1.0  0.0     0.0  0.006312
1     5       1.0  0.0     NaN  0.006312
2     5       1.0  0.0     0.0  0.006312

Mean:
fold        5.000000
accuracy    1.000000
f1          0.000000
auc_pr      0.000000
brier       0.006312
dtype: float64


In [19]:
print(
    df.groupby("report_month")["target_cost_overrun_12m"]
      .agg(
          total="count",
          positives=lambda x: (x == 1).sum(),
          negatives=lambda x: (x == 0).sum()
      )
      .tail(15)
)

              total  positives  negatives
report_month                             
2021-08-01      135         13        122
2021-09-01      123         12        111
2021-10-01        0          0          0
2021-11-01        0          0          0
2021-12-01        0          0          0
2022-01-01        0          0          0
2022-02-01        0          0          0
2022-03-01        0          0          0
2022-04-01        0          0          0
2022-05-01        0          0          0
2022-06-01        0          0          0
2022-07-01        0          0          0
2023-11-01        0          0          0
2024-04-01        0          0          0
2024-05-01        0          0          0


In [20]:
print(
    df.groupby("report_month")["target_schedule_risk_12m"]
      .agg(
          total="count",
          positives=lambda x: (x == 1).sum(),
          negatives=lambda x: (x == 0).sum()
      )
      .tail(15)
)

              total  positives  negatives
report_month                             
2021-08-01        0          0          0
2021-09-01        0          0          0
2021-10-01        0          0          0
2021-11-01        0          0          0
2021-12-01        0          0          0
2022-01-01        0          0          0
2022-02-01        0          0          0
2022-03-01        0          0          0
2022-04-01        0          0          0
2022-05-01        0          0          0
2022-06-01        0          0          0
2022-07-01        0          0          0
2023-11-01        0          0          0
2024-04-01        0          0          0
2024-05-01        0          0          0


In [21]:
print(df.groupby("report_month")["target_cost_overrun_12m"].agg(
    count="count",
    positives=lambda x: (x == 1).sum(),
    negatives=lambda x: (x == 0).sum()
).tail(40))

              count  positives  negatives
report_month                             
2019-03-01     1185        133       1052
2019-04-01     1235        134       1101
2019-05-01     1483        128       1355
2019-06-01     1506         98       1408
2019-07-01     1537        112       1425
2019-08-01     1568         99       1469
2019-09-01     1546         97       1449
2019-10-01     1543         89       1454
2019-11-01     1606        110       1496
2019-12-01     1603        101       1502
2020-01-01     1608        108       1500
2020-02-01     1605        112       1493
2020-03-01     1582        123       1459
2020-04-01     1579        131       1448
2020-05-01     1588        133       1455
2020-06-01     1589        146       1443
2020-07-01     1569        114       1455
2020-08-01     1578        110       1468
2020-09-01     1582        117       1465
2020-12-01     1419         96       1323
2021-01-01     1484         94       1390
2021-02-01     1489        130    

In [22]:
print(df.groupby("report_month")["target_schedule_risk_12m"].agg(
    count="count",
    positives=lambda x: (x == 1).sum(),
    negatives=lambda x: (x == 0).sum()
).tail(40))

              count  positives  negatives
report_month                             
2019-03-01      972        280        692
2019-04-01     1018        248        770
2019-05-01     1254        273        981
2019-06-01     1316        276       1040
2019-07-01     1326        322       1004
2019-08-01     1338        351        987
2019-09-01     1337        357        980
2019-10-01        0          0          0
2019-11-01        0          0          0
2019-12-01     1417        398       1019
2020-01-01     1395        394       1001
2020-02-01     1374        396        978
2020-03-01     1369        404        965
2020-04-01     1378        369       1009
2020-05-01        0          0          0
2020-06-01        0          0          0
2020-07-01     1335        306       1029
2020-08-01     1261        298        963
2020-09-01     1199        297        902
2020-12-01     1212        330        882
2021-01-01     1280        346        934
2021-02-01     1189        414    